# 01. XAI Local Surrogates: LIME, SHAP, and Perturbation Masks

- **Category:** 10_xai_and_ai_analysis_methods
- **Lineage & Citations:** Shapley (1953 *A Value for n-person Games*); Ribeiro et al. (2016 *Why Should I Trust You? LIME*); Lundberg & Lee (2017 *A Unified Approach to Interpreting Model Predictions: KernelSHAP*); Petsiuk et al. (2018 *RISE*); Hooker et al. (2019 *A Benchmark for Interpretability Methods: ROAR*).
- **Purpose:** Construct local surrogate models (LIME, KernelSHAP) and randomized perturbation masks (RISE) to explain black-box model decisions and audit explanation faithfulness via ROAR.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [09. Causal AI & Evolutionary Search](../09_causal_ai_and_evolutionary_search/07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.ipynb) $\to$ `🔗 Downstream Syntheses:` [02. XAI Gradient Propagation IG and LRP](02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use independent marginal perturbation on strongly correlated features (creates out-of-distribution inputs that trick black boxes); avoid unverified saliency maps that act as cosmetic edge detectors rather than faithful reflections of decision logic.

## Mathematical Foundations & Governing Equations

1. **LIME Weighted Ridge Surrogate**: $w^* = (Z'^T \Pi Z' + \lambda I)^{-1} Z'^T \Pi y, \quad \Pi = \text{diag}(\pi_1, \dots, \pi_N)$
2. **KernelSHAP Combinatorial Weights**: $\pi(z') = \frac{M - 1}{\binom{M}{|z'|} |z'| (M - |z'|)}$
3. **RISE Feature Attribution**: $S_i = \frac{1}{N} \sum_{k=1}^N f(x \odot m_k) \cdot m_{k, i}$
4. **Faithfulness Degradation (ROAR)**: $\Delta y_k = y_{\text{orig}} - f(x \odot (1 - \text{mask}_{\text{top-}k}))$

In [ ]:
import sys, time, math
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/10_xai_and_ai_analysis_methods/01_xai_local_surrogates_lime_shap_and_perturbation_masks.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def lime_regression(z_prime, y, pi, reg=1e-4):
    sqrt_pi = torch.sqrt(pi).unsqueeze(-1)
    z_w = z_prime * sqrt_pi
    y_w = y * sqrt_pi.squeeze(-1)
    lhs = z_w.t() @ z_w + reg * torch.eye(z_prime.shape[1], dtype=z_prime.dtype, device=z_prime.device)
    return torch.linalg.solve(lhs, z_w.t() @ y_w)

def shap_weights(coalition_sizes, m_features=4):
    res = torch.zeros_like(coalition_sizes)
    for i in range(coalition_sizes.shape[0]):
        s = int(coalition_sizes[i].item())
        if 0 < s < m_features:
            binom = math.comb(m_features, s)
            res[i] = float(m_features - 1) / (float(binom) * float(s) * float(m_features - s))
        else: res[i] = 1000.0
    return res

def rise_importance(masks, model_outputs):
    return (model_outputs.unsqueeze(0) @ masks).squeeze(0) / float(masks.shape[0])

def roar_degrade(y_orig, degraded_y):
    return y_orig - degraded_y

In [ ]:
# Parity verification against golden reference tensors
w_lime = lime_regression(weights["lime_z"], weights["lime_y"], weights["lime_pi"], reg=1e-4)
assert torch.max(torch.abs(w_lime - weights["lime_w_ref"])) < 1e-5, "LIME weights mismatch"

w_shap = shap_weights(weights["shap_sizes"], m_features=4)
assert torch.max(torch.abs(w_shap - weights["shap_weights_ref"])) < 1e-5, "SHAP weights mismatch"

imp = rise_importance(weights["rise_masks"], weights["rise_outputs"])
assert torch.max(torch.abs(imp - weights["rise_importance_ref"])) < 1e-5, "RISE importance mismatch"

deg = roar_degrade(weights["roar_orig"], weights["roar_degraded"])
assert torch.max(torch.abs(deg - weights["roar_degradation_ref"])) < 1e-5, "ROAR degradation mismatch"
print("All LIME, SHAP, RISE, and ROAR parity assertions passed!")

In [ ]:
# Benchmarking local surrogate and perturbation operators
t0 = time.perf_counter()
for _ in range(500):
    _ = lime_regression(weights["lime_z"], weights["lime_y"], weights["lime_pi"], reg=1e-4)
    _ = shap_weights(weights["shap_sizes"], m_features=4)
    _ = rise_importance(weights["rise_masks"], weights["rise_outputs"])
    _ = roar_degrade(weights["roar_orig"], weights["roar_degraded"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused LIME + KernelSHAP + RISE latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Cosmetic Saliency Trap audited via ROAR Degradation
# Model depends heavily on feature 0 and 1, completely invariant to feature 2 and 3
def model_fn(x): return 2.0 * x[0] + 3.0 * x[1]
x_sample = torch.tensor([1.0, 1.0, 5.0, 5.0])
y_true = model_fn(x_sample).item()
# Faithful explanation removes true salient features (feat 1, then feat 0)
x_faithful = x_sample.clone(); x_faithful[1] = 0.0
deg_faithful = y_true - model_fn(x_faithful)
# Cosmetic artifact removes visually high-magnitude noise feature (feat 2)
x_cosmetic = x_sample.clone(); x_cosmetic[2] = 0.0
deg_cosmetic = y_true - model_fn(x_cosmetic)
print(f"Faithful Degradation: {deg_faithful:.2f} vs Cosmetic Degradation: {deg_cosmetic:.2f}")
assert deg_faithful > 2.0 and deg_cosmetic == 0.0, "ROAR confirms cosmetic explanations fail to degrade model output!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **LIME** fits distance-weighted linear surrogates on perturbation neighborhoods.
2. **KernelSHAP** selects unique combinatorial weights guaranteeing exact Shapley value equivalence.
3. **ROAR Auditing** verifies whether saliency maps reflect true decision logic rather than cosmetic edge detection.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_xai_local_surrogates_lime_shap_and_perturbation_masks.md](../../markdown/10_xai_and_ai_analysis_methods/01_xai_local_surrogates_lime_shap_and_perturbation_masks.md)
- **PyTorch 2.x Module:** [01_xai_local_surrogates_lime_shap_and_perturbation_masks.py](../../pytorch2/10_xai_and_ai_analysis_methods/01_xai_local_surrogates_lime_shap_and_perturbation_masks.py)
- **Functional JAX Module:** [01_xai_local_surrogates_lime_shap_and_perturbation_masks.py](../../jax/10_xai_and_ai_analysis_methods/01_xai_local_surrogates_lime_shap_and_perturbation_masks.py)
- **Native Rust Candle Track:** [01_xai_local_surrogates_lime_shap_and_perturbation_masks.rs](../../candle/10_xai_and_ai_analysis_methods/01_xai_local_surrogates_lime_shap_and_perturbation_masks.rs)
- **Hardware Kernel (CPU / WGSL):** [01_xai_local_surrogates_lime_shap_and_perturbation_masks.rs](../../kernels/10_xai_and_ai_analysis_methods/01_xai_local_surrogates_lime_shap_and_perturbation_masks.rs)
- **Golden SafeTensors:** [01_xai_local_surrogates_lime_shap_and_perturbation_masks.safetensors](../../weights/10_xai_and_ai_analysis_methods/01_xai_local_surrogates_lime_shap_and_perturbation_masks.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 09/07 Causal Representation Identifiability iVAE Synthetic Control and Symbolic Sr](../09_causal_ai_and_evolutionary_search/07_causal_representation_identifiability_ivae_synthetic_control_and_symbolic_sr.ipynb) | [Master Index](../../README.md#10_xai_and_ai_analysis_methods) | [Next: 02. XAI Gradient Propagation IG LRP DeepLIFT Grad-CAM and Attention](02_xai_gradient_propagation_ig_lrp_deeplift_gradcam_and_attention.ipynb) &rarr;